## Directory

In [5]:
import polars as pl
import os
import geopandas as gpd
import numpy as np
import pandas as pd
import csv

main_dir = os.path.dirname(os.path.dirname(os.getcwd()))

### Tweet work

In [6]:
rbarrera = r"C:\Users\rjbar\Saadgulzar Dropbox\rbarreraf@fen.uchile.cl\sa_fires"
aquisper = '/Users/anzony.quisperojas/Library/CloudStorage/Dropbox/sa_fires'
main = aquisper
input_data = f"{main}/data/input/politician_opinions/twitter"

In [7]:
df = pd.read_csv(
    fr"{input_data}/scraped_tweets_complete2.csv",
    low_memory=False
)[['url','text','lang','handle_searched','createdAt','isRetweet']].rename(columns={'createdAt':'date'})

df = df.loc[(df['isRetweet']=='False')|(df['isRetweet']=='0.0')]

In [8]:
translated = pd.concat([
    pd.read_csv(os.path.join(main,'data','interim','politician_opinions','twitter','translated_tweets_hindi_partial2.csv')),
    pd.read_csv(os.path.join(main,'data','interim','politician_opinions','twitter','translated_tweets_hindi_partial.csv')),
    pd.read_csv(os.path.join(main,'data','interim','politician_opinions','twitter','translated_tweets_hindi_partial_aux.csv'))    
    ]).drop_duplicates(['original', 'translated'])[['original','translated']]

translated = translated.loc[translated['translated'].notna()].rename(columns={'original':'text'})

In [9]:
df = df.merge(translated, how='left', on='text')
    
df['text'] = np.where(
    ((df['translated'].notna())&(df['lang']!='en')), df['translated'], df['text']
)

del(translated)

In [10]:
df_analysis_1 = df.merge(pd.read_csv(
    fr"{main}/data/interim/politician_opinions/twitter/analyse_tweets.csv",
    low_memory=False).rename(columns={'Unnamed: 0':'tweet_id'}), how='left', on=['text','date'])

df_analysis_2 = df_analysis_1.loc[df_analysis_1['tweet_id'].isna()].drop(columns=['tweet_id'])
df_analysis_2 = df_analysis_2.merge(
    pd.read_csv(fr"{main}/data/interim/politician_opinions/twitter/analyse_tweets.csv").drop(columns=['date']).rename(columns={'Unnamed: 0':'tweet_id'}).drop_duplicates(['text']),
    how='left', on='text')[['url',  'lang', 'text', 'handle_searched', 'date', 'tweet_id','date']]

df_analysis_1 = df_analysis_1.loc[df_analysis_1['tweet_id'].notna()][['url', 'lang', 'text', 'handle_searched', 'date', 'tweet_id','date']]

In [11]:
df_analysis = pd.concat([df_analysis_1, df_analysis_2])
del (df_analysis_1, df_analysis_2, df)

In [12]:
df_analysis['date_real'] = pd.to_datetime(
    df_analysis['date'].iloc[:, 0], 
    errors='coerce'
)

df_analysis['date'] = df_analysis['date'].iloc[:, 0]

/var/folders/gb/56xms55d53n734tnp7xq55bh0000gq/T/ipykernel_7472/1195123721.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_analysis['date_real'] = pd.to_datetime(


In [13]:
df_analysis['handle_searched'] = df_analysis['handle_searched'].str.replace('@', '').str.lower()
df_analysis['handle_searched'] = df_analysis['handle_searched'].str.split('?').str[0]
df_analysis['handle_searched'] = df_analysis['handle_searched'].str.replace(' ','')

df_analysis = df_analysis.drop_duplicates(['text','date_real','handle_searched'])
df_analysis = df_analysis.loc[df_analysis['text'].notna()]
print(len(df_analysis.loc[df_analysis['date_real'].isna()]))
print(len(df_analysis.loc[df_analysis['date_real'].notna()]))

0
303709


In [14]:
df_analysis.lang.value_counts(normalize=True)
#zxx

lang
hi     0.742367
pa     0.099752
en     0.075492
zxx    0.044910
qme    0.009664
und    0.006682
ne     0.004975
mr     0.004336
kn     0.002889
qht    0.001691
te     0.001589
in     0.001499
et     0.000774
qam    0.000626
art    0.000530
tl     0.000516
pl     0.000255
es     0.000136
or     0.000126
ml     0.000106
ht     0.000089
bn     0.000079
it     0.000076
ca     0.000076
ta     0.000076
ro     0.000076
pt     0.000070
fr     0.000066
da     0.000056
tr     0.000053
qst    0.000046
nl     0.000040
sv     0.000036
fi     0.000033
gu     0.000030
no     0.000026
eu     0.000026
de     0.000023
sl     0.000020
ur     0.000020
is     0.000013
hu     0.000013
vi     0.000010
cy     0.000010
lt     0.000007
ar     0.000007
lv     0.000003
cs     0.000003
Name: proportion, dtype: float64

In [36]:
official =pd.read_excel(os.path.join(main, "data", "input", "politician_opinions", "official_twitter.xlsx"))
official_2 = pd.read_excel(os.path.join(main, "data", "input", "politician_opinions", "official_social_profiles.xlsx"))
official_2 = official_2.loc[official_2['platform']== 'X (Twitter)']
official_2['username'] = official_2['url'].str.split('/').str[-1]
official = official.merge(official_2, how='outer', left_on='Politician Name', right_on='name')
official['Twitter Handle'] = official['Twitter Handle'].fillna(official['username']).str.replace("@", "").str.lower()
official = official.loc[official['Twitter Handle'].notna()]

official['Twitter Handle'] = official['Twitter Handle'].str.replace('@', '').str.lower()
official['Twitter Handle'] = official['Twitter Handle'].str.split('?').str[0]
official['Twitter Handle'] = official['Twitter Handle'].str.replace(' ','')
official['handle_searched'] = official['Twitter Handle']

In [41]:
print('len pre merge', len(official))
twinfo = df_analysis.merge(official , how = 'left', on = 'handle_searched', indicator=True)
print('len post merge', len(twinfo))
print(twinfo['_merge'].value_counts())

len pre merge 1591
len post merge 352723
_merge
both          351153
left_only       1570
right_only         0
Name: count, dtype: int64


In [18]:
len(twinfo.loc[twinfo['_merge']=='left_only'].drop_duplicates('handle_searched'))

1

In [19]:
twinfo['Politician Name'] = (
    twinfo
    .groupby('handle_searched')['Politician Name']
    .transform(lambda s: s.ffill().bfill())
)

In [20]:
twinfo['month'] = pd.to_datetime(twinfo['date_real']).dt.month
twinfo['year'] = pd.to_datetime(twinfo['date_real']).dt.year

In [21]:
print('len post merge', len(twinfo.drop_duplicates('Politician Name')))

len post merge 705


In [23]:
# Winners universe: all winners since 2008.
winners = pd.read_csv(os.path.join(main, "data", "input", "my_neta", "2008_onwards_winners_table.csv"))
winners['ASSEMBLY']      = winners['unique_id'].str.split("_", expand=True).iloc[:, 1].astype(float)
winners['election_year'] = winners['unique_id'].str.split("_", expand=True).iloc[:, 0].str[-4:].astype(float)
winners['STATE_UT']      = winners['unique_id'].str.split("_", expand=True).iloc[:, 0].str[:-4].str.upper()
winners['STATE_UT'] = winners['STATE_UT'].replace({
    'UTTARPRADESH':'UTTAR PRADESH','UP':'UTTAR PRADESH',
    'HA':'HARYANA','BIH':'BIHAR','PB':'PUNJAB'
})

# politician_id = lowered name with spaces removed. Same id can span multiple cycles
# (re-elections), but only if all those cycles are in the SAME state. If the same name
# appears in >1 state, those rows almost certainly correspond to different physical
# politicians and we cannot tell from the Twitter file (no state on that side) which
# one a given handle belongs to, so we drop them.
winners['politician_id'] = winners['name'].str.lower().str.replace(' ', '')
state_count   = winners.groupby('politician_id')['STATE_UT'].nunique()
ambiguous_ids = set(state_count[state_count > 1].index)
print(f'Cross-state name collisions dropped: {len(ambiguous_ids)} politicians '
      f'({winners["politician_id"].isin(ambiguous_ids).sum()} cycle rows)')

winners_clean = winners[~winners['politician_id'].isin(ambiguous_ids)].copy()
print(f'After disambiguation: {winners_clean["politician_id"].nunique()} politicians, '
      f'{winners_clean["unique_id"].nunique()} cycles')

Cross-state name collisions dropped: 30 politicians (88 cycle rows)
After disambiguation: 2081 politicians, 2537 cycles


In [24]:
print(winners.STATE_UT.value_counts())
print('number of politicians ',len(winners.drop_duplicates('name')))
print('number of election id ', len(winners.drop_duplicates('unique_id')))

STATE_UT
UTTAR PRADESH    1209
BIHAR             707
HARYANA           358
PUNJAB            351
Name: count, dtype: int64
number of politicians  2198
number of election id  2625


In [26]:
shp_path   = os.path.join(main, 'proj_bureaucrats_farms', "data_output", "intermediate", "_0_2_3_ACs_right_shapefile.shp")
ac_shp     = gpd.read_file(shp_path)
panel_data = pd.read_stata(os.path.join(main, 'proj_bureaucrats_farms', "data_output", "intermediate", "panel_data_election_year.dta"))

In [27]:
# Apply the same politician_id to twinfo and drop cross-state-ambiguous handles.
twinfo['politician_id'] = twinfo['Politician Name'].str.lower().str.replace(' ', '')
n_before = twinfo['politician_id'].nunique()
twinfo   = twinfo[twinfo['politician_id'].notna() & ~twinfo['politician_id'].isin(ambiguous_ids)].copy()
print(f'Twitter politicians: {n_before} pre-filter -> {twinfo["politician_id"].nunique()} after disambiguation')

# Cycle info: one row per (politician_id, unique_id) carrying take/end dates from panel_data.
cycle_info = (
    winners_clean[['politician_id', 'name', 'STATE_UT', 'unique_id', 'ac_name']]
    .merge(
        panel_data[['unique_id','ac_uq_id','election_year','month_take','year_take','month_end','year_end']]
        .drop_duplicates('unique_id'),
        on='unique_id', how='left'
    )
    .dropna(subset=['month_take','year_take','month_end','year_end'])
    .copy()
)
print(f'Cycles with valid take/end dates: {len(cycle_info)}')

Twitter politicians: 694 pre-filter -> 678 after disambiguation
Cycles with valid take/end dates: 2537


## Rubric 2 DF (agriculture & farmers)

In [ ]:
# Load rubric_2 (agriculture & farmers classifications) and clean / normalize values.
# Allowed values come from rubric_2_agriculture_farmers.md. Anything outside the
# allowed set is set to NaN so the pivot only generates the outcome columns we want.

rubric_2 = pd.read_csv(
    fr"{main}/data/interim/politician_opinions/twitter/rubric_2_output.csv",
    engine='python',
    quoting=csv.QUOTE_ALL,
    on_bad_lines='warn'
)

FRAMING_VALID         = {'provider_hero','victim','protester_activist','beneficiary',
                         'problem_causer','neutral_reference','rhetorical'}
STANCE_FARMERS_VALID  = {'pro_farmer','anti_farmer','neutral','not_applicable'}
STANCE_LAWS_VALID     = {'pro_laws','anti_laws','neutral','not_applicable'}
WELFARE_POLICY_MAP    = {'msp':'msp_procurement','loan_waiver':'debt_relief'}
WELFARE_POLICY_VALID  = {'pm_kisan','crop_insurance','machinery_distribution',
                         'msp_procurement','debt_relief','irrigation_scheme',
                         'soil_health','organic_farming','other_farm_policy','none'}
BLAME_VALID           = {'central_govt','state_govt','opposition','middlemen',
                         'bureaucracy','other','none'}

def _coerce_int(v):
    if pd.isna(v): return np.nan
    try:
        n = int(float(v))
        return n if 0 <= n <= 3 else np.nan
    except (TypeError, ValueError):
        return np.nan

def _norm_cat(v, valid):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    return s if s in valid else np.nan

def _norm_welfare(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    s = WELFARE_POLICY_MAP.get(s, s)
    return s if s in WELFARE_POLICY_VALID else np.nan

def _norm_credit(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    if s in ('true','1','1.0'):  return 'true'
    if s in ('false','0','0.0'): return 'false'
    return np.nan

rubric_2['agriculture_relevance']    = rubric_2['agriculture_relevance'].apply(_coerce_int)
rubric_2['farmer_framing']           = rubric_2['farmer_framing'].apply(lambda v: _norm_cat(v, FRAMING_VALID))
rubric_2['stance_farmers']           = rubric_2['stance_farmers'].apply(lambda v: _norm_cat(v, STANCE_FARMERS_VALID))
rubric_2['farmer_protest_relevance'] = rubric_2['farmer_protest_relevance'].apply(_coerce_int)
rubric_2['stance_farm_laws']         = rubric_2['stance_farm_laws'].apply(lambda v: _norm_cat(v, STANCE_LAWS_VALID))
rubric_2['farmer_welfare_policy']    = rubric_2['farmer_welfare_policy'].apply(_norm_welfare)
rubric_2['credit_claiming']          = rubric_2['credit_claiming'].apply(_norm_credit)
rubric_2['blame_attribution']        = rubric_2['blame_attribution'].apply(lambda v: _norm_cat(v, BLAME_VALID))

print('After cleaning, non-null counts per field:')
for c in ['agriculture_relevance','farmer_framing','stance_farmers','farmer_protest_relevance',
          'stance_farm_laws','farmer_welfare_policy','credit_claiming','blame_attribution']:
    print(f'  {c:30s} {rubric_2[c].notna().sum():>7}')

# Merge rubric_2 fields into twinfo.
twinfo = twinfo.merge(
    rubric_2.drop_duplicates('tweet_id')[
        ['tweet_id','agriculture_relevance','farmer_framing','stance_farmers',
         'farmer_protest_relevance','stance_farm_laws','farmer_welfare_policy',
         'credit_claiming','blame_attribution']
    ],
    how='left', on='tweet_id'
)

# rubric_2_output has no is_quoted column -- pull it from rubric_1 (same tweet_id key).
rubric_1_iq = pd.read_csv(
    fr"{main}/data/interim/politician_opinions/twitter/rubric_1_output.csv",
    engine='python', quoting=csv.QUOTE_ALL, on_bad_lines='warn',
    usecols=['tweet_id','is_quoted']
)
twinfo = twinfo.merge(rubric_1_iq.drop_duplicates('tweet_id'), how='left', on='tweet_id')

In [ ]:
# Tweet aggregation function for rubric_2. Produces a (politician_id, year, month)
# panel of tweet counts with `_<suffix>` appended to every metric column. Suffix
# is 'all' (every classified tweet) or 'own' (is_quoted == False).
#
# Column scheme (each suffixed `_<suffix>`):
#   ar0..ar3              : counts by agriculture_relevance level (0-3)
#   ff_<framing>          : counts by farmer_framing value
#   sf_<stance>           : counts by stance_farmers value
#   pr0..pr3              : counts by farmer_protest_relevance level (0-3)
#   sl_<stance>           : counts by stance_farm_laws value
#   wp_<policy>           : counts by farmer_welfare_policy value
#   cc_true, cc_false     : counts by credit_claiming value
#   ba_<target>           : counts by blame_attribution value
#
# All column names stay <= 32 chars after `_all`/`_own` is appended (Stata limit).

def aggregate_rubric2(twf, suffix):
    keys = ['politician_id','year','month']

    def pivot_numeric(col, prefix):
        """Pivot 0-3 score column into <prefix>0..<prefix>3 (no separator)."""
        sub = twf.dropna(subset=[col])
        if sub.empty:
            return pd.DataFrame(columns=keys)
        out = (sub.groupby(keys + [col])['text'].nunique()
               .reset_index().rename(columns={'text':'n'})
               .pivot_table(index=keys, columns=col, values='n', fill_value=0)
               .reset_index())
        out.columns = [c if c in keys else f'{prefix}{int(c)}' for c in out.columns]
        return out

    def pivot_string(col, prefix):
        sub = twf.dropna(subset=[col])
        if sub.empty:
            return pd.DataFrame(columns=keys)
        return (sub.groupby(keys + [col])['text'].nunique()
                .reset_index().rename(columns={'text':'n'})
                .pivot_table(index=keys, columns=col, values='n', fill_value=0)
                .add_prefix(prefix).reset_index())

    parts = [
        pivot_numeric('agriculture_relevance',    'ar'),
        pivot_string ('farmer_framing',           'ff_'),
        pivot_string ('stance_farmers',           'sf_'),
        pivot_numeric('farmer_protest_relevance', 'pr'),
        pivot_string ('stance_farm_laws',         'sl_'),
        pivot_string ('farmer_welfare_policy',    'wp_'),
        pivot_string ('credit_claiming',          'cc_'),
        pivot_string ('blame_attribution',        'ba_'),
    ]
    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on=keys, how='outer')
    out = out.fillna(0)

    rename = {c: f'{c}_{suffix}' for c in out.columns if c not in keys}
    return out.rename(columns=rename)


# Restrict to tweets that received at least one rubric_2 classification, and to
# the panel period (panel ends 2025-12).
rubric2_fields = ['agriculture_relevance','farmer_framing','stance_farmers',
                  'farmer_protest_relevance','stance_farm_laws','farmer_welfare_policy',
                  'credit_claiming','blame_attribution']
twinfo_classified = twinfo.loc[
    twinfo[rubric2_fields].notna().any(axis=1) & (twinfo['year'] <= 2025)
].copy()

agg_all = aggregate_rubric2(twinfo_classified,                                              'all')
agg_own = aggregate_rubric2(twinfo_classified[twinfo_classified['is_quoted'] == False],     'own')

print('agg_all:', agg_all.shape, '| agg_own:', agg_own.shape)

In [ ]:
# === Build per-politician (year, month) panel grid ===
# Universe: every disambiguated winner (with or without tweets).
# Start  : first tweet month if the politician has any tweet, else earliest cycle take month.
# End    : 2025-12 (fixed).

universe = winners_clean[['politician_id','name']].drop_duplicates('politician_id').copy()

first_tw = (
    twinfo.dropna(subset=['date_real'])
          .assign(ym=lambda d: d['year'].astype(int) * 100 + d['month'].astype(int))
          .groupby('politician_id')['ym'].min()
          .reset_index().rename(columns={'ym':'first_tw_ym'})
)
first_take = (
    cycle_info.assign(ym=lambda d: d['year_take'].astype(int) * 100 + d['month_take'].astype(int))
              .groupby('politician_id')['ym'].min()
              .reset_index().rename(columns={'ym':'first_take_ym'})
)
universe = (universe
            .merge(first_tw,   on='politician_id', how='left')
            .merge(first_take, on='politician_id', how='left'))
universe['start_ym'] = universe['first_tw_ym'].fillna(universe['first_take_ym']).astype('Int64')
universe['end_ym']   = 202512
universe = universe.dropna(subset=['start_ym']).copy()
universe['start_ym'] = universe['start_ym'].astype(int)
print(f'Universe: {len(universe)} politicians '
      f'({universe["first_tw_ym"].notna().sum()} with tweets, '
      f'{universe["first_tw_ym"].isna().sum()} fall back to earliest cycle)')

def _months_between(start_ym, end_ym):
    s = pd.Timestamp(year=start_ym // 100, month=start_ym % 100, day=1)
    e = pd.Timestamp(year=end_ym   // 100, month=end_ym   % 100, day=1)
    rng = pd.date_range(s, e, freq='MS')
    return pd.DataFrame({'year': rng.year.astype(int), 'month': rng.month.astype(int)})

frames = []
for r in universe.itertuples(index=False):
    m = _months_between(r.start_ym, r.end_ym)
    m['politician_id'] = r.politician_id
    frames.append(m)
panel = pd.concat(frames, ignore_index=True)
print(f'Panel rows: {len(panel):,}')

# === Elected status & active-cycle metadata ===
panel['ym'] = panel['year'] * 100 + panel['month']
ci = cycle_info.copy()
ci['take_ym'] = ci['year_take'].astype(int) * 100 + ci['month_take'].astype(int)
ci['end_ym']  = ci['year_end'].astype(int)  * 100 + ci['month_end'].astype(int)

active = panel.merge(ci, on='politician_id', how='left')
active = active[(active['ym'] >= active['take_ym']) & (active['ym'] <= active['end_ym'])].copy()
# If overlapping cycles ever happen, keep the one with the latest take date.
active = (active.sort_values(['politician_id','year','month','take_ym'])
                .drop_duplicates(['politician_id','year','month'], keep='last'))
active = active[['politician_id','year','month','ac_uq_id','unique_id','election_year',
                 'month_take','year_take','month_end','year_end','ac_name']]
active['elected'] = 1

panel = panel.drop(columns=['ym']).merge(active, on=['politician_id','year','month'], how='left')
panel['elected'] = panel['elected'].fillna(0).astype(int)

# === Time-invariant politician info ===
char_rename = {
    'dependent_1_owns_agricultural_assets':'dep1_owns_agri',
    'dependent_2_owns_agricultural_assets':'dep2_owns_agri',
    'dependent_3_owns_agricultural_assets':'dep3_owns_agri',
    'self_owns_agricultural_assets':       'self_owns_agri',
    'spouse_owns_agricultural_assets':     'spouse_owns_agri',
}
char_cols = ['education','self_profession','spouse_profession'] + list(char_rename.values())
pol_info = (winners_clean
            .sort_values(['politician_id','election_year'])
            .drop_duplicates('politician_id', keep='last')
            .rename(columns=char_rename)
            [['politician_id','name','STATE_UT'] + char_cols])
panel = panel.merge(pol_info, on='politician_id', how='left')

# === Tweet aggregations (_all, _own) ===
panel = panel.merge(agg_all, on=['politician_id','year','month'], how='left')
panel = panel.merge(agg_own, on=['politician_id','year','month'], how='left')

# Fill 0 for every tweet-count column added by agg_all / agg_own.
agg_cols = sorted(set(c for c in list(agg_all.columns) + list(agg_own.columns)
                      if c not in ('politician_id','year','month')))
agg_cols = [c for c in agg_cols if c in panel.columns]
panel[agg_cols] = panel[agg_cols].fillna(0)

panel = panel.rename(columns={'politician_id':'Politician_Name'})
print(panel.shape, '| elected:', panel['elected'].value_counts().to_dict())
panel.head(3)

In [ ]:
out_path = fr'{main}/proj_bureaucrats_farms/data_output/intermediate/tweets_by_rubric2.dta'
panel.to_stata(out_path, write_index=False)
print('Saved:', out_path)